# Study object matching

Examines the all-catalog object match: SNR by catalog coverage and spatial distribution of fully-matched vs partially-matched objects.

In [ ]:
# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import tables_io
import hpmcm

## Configuration

In [ ]:
tract = 8006

In [ ]:
# Input catalog files and IDs for the object meta-match
all_object = tables_io.read(f'{tract}/object_reduced_{tract}.parq')
all_anacal = tables_io.read(f'{tract}/anacal_reduced_{tract}.parq')
all_desc_md = tables_io.read(f'{tract}/desc_md_match_object_stats.pq')
all_rubin_md = tables_io.read(f'{tract}/rubin_md_match_object_stats.pq')
obj_stats = tables_io.read(f'{tract}/all_match_object_stats.pq')
obj_assoc = tables_io.read(f'{tract}/all_match_object_assoc.pq')

all_files = [
    f'{tract}/object_reduced_{tract}.parq',
    f'{tract}/anacal_reduced_{tract}.parq',
    f'{tract}/desc_md_match_object_stats.pq',
    f'{tract}/rubin_md_match_object_stats.pq',
]
all_ids = list(range(len(all_files)))


In [ ]:
# Build wide joined table
joined_shear_obj = hpmcm.output_tables.buildJoinedObjectTable(
    obj_stats,
    obj_assoc,
    all_files,
    all_ids,
)

In [ ]:
# SNR distributions split by catalog detection bits
bins = np.logspace(0, 5, 51)
for i in range(4):
    mask_bit = 1 << i
    mask = np.bitwise_and(joined_shear_obj.catalog_mask, mask_bit) == mask_bit
    _ = plt.hist(joined_shear_obj[mask].snr, bins=bins, alpha=0.1)
_ = plt.xscale('log')
_ = plt.yscale('log')

In [ ]:
_ = plt.hist2d(joined_shear_obj.catalog_mask, joined_shear_obj.snr, bins=(np.linspace(0.5, 15.5, 16), np.logspace(0, 5, 51)), norm='log')
_ = plt.yscale('log')

In [ ]:
# Objects detected in all 4 catalogs have catalog_mask == 15
good_mask = joined_shear_obj.catalog_mask == 15
bad_mask = joined_shear_obj.catalog_mask != 15
good_sub = joined_shear_obj[good_mask]
bad_sub = joined_shear_obj[bad_mask]

In [ ]:
_ = plt.scatter(good_sub.x_cent[::10], good_sub.y_cent[::10], s=1)
_ = plt.scatter(bad_sub.x_cent[::100], bad_sub.y_cent[::100], s=1)

In [ ]:
_ = plt.scatter(joined_shear_obj.x_cent[::100], joined_shear_obj.y_cent[::100], s=1)
